# Single-Cell Analysis NORMAL TISSUE  – June 2026 

End-to-end preprocessing pipeline for paired **GEX + TCR + ADT** data from
sorted CD8 TIL samples. 

**Outline**
1. Imports
2. Sample info
3. Data loading (GEX + TCR per sample)
4. MuData assembly
5. Gene-ID annotation (mouse, Ensembl)
6. ADT modality extraction
7. Filtering genes
8. Quality-control metrics
9. Normalisation & HVG selection
10. Dimensionality reduction & clustering
11. Metadata annotation
12. AIRR-based subsetting & final save
13. Save mdata

## 1 · Imports

In [ ]:
# Standard library
import os
from pathlib import Path
from functools import partial

# Numerical / data
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from scipy.stats import median_abs_deviation

# Single-cell ecosystem
import anndata as ad
import mudata as mu
import scanpy as sc
import scirpy as ir

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns
import altair as alt

# Optional / downstream
import decoupler as dc


alt.data_transformers.enable('vegafusion')

sc.settings.verbosity = 1 

## 2 · Sample info

Define all samples with their experimental group and treatment condition.
TIL (tumour-infiltrating lymphocyte) 2021 cohort.

In [ ]:
samples = {
    "10mix1": {"group": "ctrl",},
    "10mix1": {"group": "ctrl"},
    "11mix1": {"group": "effector"},
    "11mix1": {"group": "effector"},
    "GF1": {"group": "GF"},
    "GF2": {"group": "GF"},
}   

## 3 · Data Loading

Read one **GEX** (gene expression) and one **TCR** (VDJ) AnnData per sample
from the CellRanger multi output directories.

In [ ]:
samples = {
    "10mix1": {"group": "ctrl"},
    "10mix2": {"group": "ctrl"},
    "11mix1": {"group": "effector"},
    "11mix2": {"group": "effector"},
    "GF1": {"group": "GF"},
    "GF2": {"group": "GF"},
}

adatas_tcr_2019 = {}
adatas_gex_2019 = {}

for sample, sample_meta in samples.items():

    adata_gex = sc.read_10x_h5(
        f"/data/projects/2021/MicrobialMetabolites/single-cell-sorted-cd8/2019-10-29_sorted_cd8/analyses_icbi/{sample}/outs/per_sample_outs/{sample}/count/sample_filtered_feature_bc_matrix.h5",
        gex_only=False,
    )

    adata_tcr = ir.io.read_10x_vdj(
        f"/data/projects/2021/MicrobialMetabolites/single-cell-sorted-cd8/2019-10-29_sorted_cd8/analyses_icbi/{sample}/outs/multi/vdj_t/all_contig_annotations.csv"
    )

    adata_gex.var_names_make_unique()

    adatas_tcr_2019[sample] = adata_tcr
    adatas_gex_2019[sample] = adata_gex

### Concatenate per-sample AnnData objects

In [ ]:
adata_tcr_2019 = ad.concat(adatas_tcr_2019, index_unique="_")

In [ ]:
adata_gex_2019 = ad.concat(adatas_gex_2019, index_unique="_")

## 4 · MuData Assembly

Combine GEX and TCR (AIRR) into a single `MuData` container.

In [ ]:
mdata = mu.MuData({"gex": adata_gex_2019, "airr": adata_tcr_2019})

## 5 · Gene-ID Annotation

Map gene symbols → Ensembl IDs (mouse / mm10) using **mygene**.
This is required for downstream tools that expect Ensembl IDs.

In [ ]:
import mygene
import pandas as pd

mg = mygene.MyGeneInfo()

genes = mdata["gex"].var_names.tolist()

res = mg.querymany(
    genes,
    scopes="symbol",
    fields="ensembl.gene",
    species="mouse",
    as_dataframe=True
)

res = res.reset_index().rename(columns={"query": "gene_symbol"})

res.columns

In [ ]:
# Case 1: if column is called "ensembl.gene"
if "ensembl.gene" in res.columns:
    mapping = (
        res.dropna(subset=["ensembl.gene"])
        .drop_duplicates("gene_symbol")
        .set_index("gene_symbol")["ensembl.gene"]
    )

# Case 2: if column is called "ensembl"
elif "ensembl" in res.columns:
    def extract_ensembl(x):
        if isinstance(x, dict):
            return x.get("gene")
        elif isinstance(x, list):
            return x[0].get("gene") if len(x) > 0 else None
        else:
            return None

    res["gene_ids"] = res["ensembl"].apply(extract_ensembl)

    mapping = (
        res.dropna(subset=["gene_ids"])
        .drop_duplicates("gene_symbol")
        .set_index("gene_symbol")["gene_ids"]
    )

In [ ]:
mdata["gex"].var["gene_ids"] = mdata["gex"].var_names.map(mapping)
mdata["gex"].var["feature_types"] = "Gene Expression"

## 6 · ADT Modality Extraction

Split antibody-capture (ADT) features into a dedicated `mdata['adt']` modality.

In [ ]:
mask = (
    mdata["gex"].var["gene_ids"].isna() &
    mdata["gex"].var.index.str.contains("_TotalSeqC")
)

mdata["gex"].var.loc[mask, "feature_types"] = "Antibody Capture"

In [ ]:
mdata.mod["adt"] = mdata["gex"][
    :,
    mdata["gex"].var["feature_types"] == "Antibody Capture"
].copy()

In [ ]:
mdata.update()

In [ ]:
mdata.update()

In [ ]:
def update_columns_condition(row):
    
    if row["sample_id"] == "GF1":
        row["condition"] = "GF"
    elif row["sample_id"] == "GF2":
        row["condition"] = "GF"
    elif row["sample_id"] == "10mix1":
        row["condition"] = "ctrl"
    elif row["sample_id"] == "10mix2":
        row["condition"] = "ctrl"
    elif row["sample_id"] == "11mix1":
        row["condition"] = "effector"
    elif row["sample_id"] == "11mix2":
        row["condition"] = "effector"

    return row

In [ ]:
mdata["gex"].obs["sample_id"] = (
    mdata["gex"].obs_names
    .str.split("_")
    .str[-1]
)

In [ ]:
mdata["gex"].obs = mdata["gex"].obs.apply(update_columns_condition, axis=1)

In [ ]:
mdata["gex"].obs["sample_id"] = mdata["gex"].obs["sample_id"].replace({
    "10mix1": "ctrl1",
    "10mix2": "ctrl2",
    "11mix1": "effector1",
    "11mix2": "effector2",
})

In [ ]:
mdata["gex"].obs

## 9 · Quality-Control Metrics

- Counts layer contains raw counts

Flag mitochondrial, ribosomal, and haemoglobin genes,
then compute per-cell QC statistics.

In [ ]:
mdata["gex"].layers["counts"] = mdata["gex"].X.copy()

In [ ]:
mdata["gex"].var["mito"] = mdata["gex"].var_names.str.startswith("mt")
mdata["gex"].var["ribo"] = mdata["gex"].var_names.str.startswith("Rp")
mdata["gex"].var["hb"] = mdata["gex"].var_names.str.startswith("hb")
sc.pp.calculate_qc_metrics(
    mdata["gex"],
    qc_vars=["mito", "ribo", "hb"],
    inplace=True,
    percent_top=[20],
    log1p=True,
)

In [ ]:
mdata

In [ ]:
sc.pl.violin(
    mdata["gex"],
    ["n_genes_by_counts", "total_counts", "pct_counts_mito"],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
sc.pl.violin(
    mdata["gex"],
    keys=["n_genes_by_counts", "total_counts", "pct_counts_mito"],
    groupby="sample_id",
    rotation=45,
    multi_panel=True,
)

In [ ]:
sc.pl.scatter(
    mdata["gex"],
    x="total_counts",
    y="n_genes_by_counts",
    color="pct_counts_mito",
)

sc.pl.scatter(
    mdata["gex"],
    x="total_counts",
    y="pct_counts_mito",
)

## 10 · Normalisation & Highly Variable Genes

Normalise library size, log-transform, and select the top 1000 highly_variable_genes

### Create counts layer with RAW counts

In [ ]:
# Saving count data
mdata["gex"].layers["counts"] = mdata["gex"].X.copy()

In [ ]:
sc.pp.normalize_total(mdata["gex"])
sc.pp.log1p(mdata["gex"])

In [ ]:
sc.pp.highly_variable_genes(mdata["gex"], n_top_genes=1000, batch_key="sample_id")

In [ ]:
sc.tl.pca(mdata["gex"])

In [ ]:
sc.pl.pca_variance_ratio(mdata["gex"], n_pcs=50, log=True)

In [ ]:
sc.pl.pca(
    mdata["gex"],
    color=["sample_id",  "pct_counts_mito"],
    dimensions=[(0, 1), (2, 3)],
    ncols=2,
    size=2,
)

## 7 · Filtering genes

In [ ]:
sc.pp.filter_cells(mdata["gex"], min_genes=100)
sc.pp.filter_genes(mdata["gex"], min_cells=3)

## 11 · Dimensionality Reduction & Leiden Clustering

Build a kNN graph, embed with UMAP, and compute Leiden clusters at three
resolutions for exploratory inspection.

In [ ]:
sc.pp.neighbors(mdata["gex"])

In [ ]:
sc.tl.umap(mdata["gex"])

In [ ]:
sc.pl.umap(mdata["gex"], color = ["sample_id","condition"])

In [ ]:
sc.tl.leiden(mdata["gex"], resolution=1, key_added="rna_leiden")

In [ ]:
sc.pl.umap(mdata["gex"], color = ["rna_leiden"])

### Filter out doublets

In [ ]:
sc.pp.scrublet(mdata["gex"], batch_key="sample_id")

In [ ]:
sc.pl.umap(
    mdata["gex"],
    color=["rna_leiden", "predicted_doublet", "doublet_score"],
    # increase horizontal space between panels
    wspace=0.5,
    size=3,
)

In [ ]:
sc.pl.umap(
    mdata["gex"],
    color=["rna_leiden", "log1p_total_counts", "pct_counts_mito", "log1p_n_genes_by_counts","predicted_doublet"],
    wspace=0.5,
    ncols=2,
)

In [ ]:
mdata["gex"].obs.predicted_doublet.value_counts()

### Visualize QC metrics

In [ ]:
mdata["adt"].obs["sample_id"] = mdata["gex"].obs["sample_id"]

In [ ]:
mdata.update()

## 12 · AIRR-Based Subsetting & Final MuData

Restrict the ADT modality to cells that also have TCR data (AIRR), then
sync the MuData object.

In [ ]:
# cells present in AIRR
airr_cells = mdata["airr"].obs_names

# subset ADT to those cells
mdata.mod["adt"] = mdata["adt"][airr_cells, :].copy()

In [ ]:
mdata.update()

## 13 · Save Mudata


In [ ]:
import pandas as pd

for mod in mdata.mod.keys():
    adata = mdata[mod]

    # fix obs columns
    for col in adata.obs.columns:
        if adata.obs[col].dtype == "object":
            adata.obs[col] = adata.obs[col].astype(str)

    # fix var columns
    for col in adata.var.columns:
        if adata.var[col].dtype == "object":
            adata.var[col] = adata.var[col].fillna("").astype(str)

mdata.update()


In [ ]:
mdata.write("/data/projects/2021/MicrobialMetabolites/single-cell-sorted-cd8/results/40_gex_surface_prot/single_cell_normal/001_create_mudata_normal.h5mu")